In [1]:
import echoshader
import xarray as xr
import panel as pn
import holoviews as hv
from holoviews import opts
from holoviews.streams import PolyDraw, PolyEdit
from shapely.geometry import Polygon
import geopandas as gpd
import numpy as np
import pandas as pd
import os
import re
import fsspec
import ast

hv.extension('bokeh')

In [ ]:
# ------------------ DATA SOURCES ------------------ #

bucket_name = "agr230002-bucket01"
csv_prefix = "hake_data/label_allocations/2017/transect_subgroup_class_dataframes/"
zarr_prefix = "hake_data/data_zarr/MVBS"
SAVE_DIR = 'region_csv/'

try:
    os.mkdir(SAVE_DIR)
except FileExistsError:
    pass

In [ ]:
# ------------------ S3 Setup ------------------ #

fs = fsspec.filesystem(
    "s3",
    anon=True,
    client_kwargs={"endpoint_url": "https://sdsc.osn.xsede.org"}
)

In [ ]:
# ------------------ Load Region Labels ------------------ #

def load_region_labels_s3(prefix, bucket=bucket_name):
    all_regions = []
    found_years = set()
    all_keys = fs.find(f"{bucket}/{prefix}")
    csv_keys = [k for k in all_keys if k.endswith(".csv")]

    for path in csv_keys:
        with fs.open(path, mode="r") as f:
            df = pd.read_csv(f)
            if not df.empty:
                file_name = path.split("/")[-1]
                df["source_file"] = file_name.replace("--all_class_info.csv", "")
                all_regions.append(df)

    if not all_regions:
        return pd.DataFrame(), []

    combined_df = pd.concat(all_regions, ignore_index=True)
    if "time" not in combined_df.columns:
        return pd.DataFrame(), []

    results = []
    for _, row in combined_df.iterrows():
        time_list = re.findall(r"'([^']+)'", row["time"])
        if time_list:
            time = pd.to_datetime(time_list)
            #time_ms = time.astype("int64") / 10**6
            found_years.update([time.min().year, time.max().year])
            results.append({
                "id": row["source_file"],
                "region_id": row["region_id"],
                "start_time": time.min(),
                "end_time": time.max(),
                #"start_time_ms": time_ms.min(),
                #"end_time_ms": time_ms.max(),
                "ping_time": row["time"],
                "depth": row["depth"]
            })
    return pd.DataFrame(results).sort_values(by="start_time").reset_index(drop=True), sorted(list(found_years))


# -------- Load MVBS Zarrs -------- #
def load_mvbs_zarr_s3(base_prefix, years, bucket=bucket_name):
    ping_records = []
    for year in years:
        prefix = f"{base_prefix}/{year}/"
        try:
            all_keys = fs.ls(f"{bucket}/{prefix}", detail=False)
        except:
            continue
        zarr_paths = [key for key in all_keys if key.endswith(".zarr")]

        for zarr_key in zarr_paths:
            try:
                ds = xr.open_zarr(fs.get_mapper(f"s3://{zarr_key}"), chunks={"ping_time": 1000})
                times = pd.to_datetime(ds['ping_time'].values)
            except:
                continue
            ping_records.append({
                'id': zarr_key.split('/')[-1].replace(".zarr", ""),
                'ping_start': times.min(),
                'ping_end': times.max(),
                #'ping_start_ms': times.min().timestamp() * 1000,
                #'ping_end_ms': times.max().timestamp() * 1000
            })
    return pd.DataFrame(ping_records).sort_values(by="ping_start").reset_index(drop=True)

region_time_df, years = load_region_labels_s3(prefix=csv_prefix)
ping_time_df = load_mvbs_zarr_s3(zarr_prefix, years)


In [ ]:
# -------- Function to find matching zarr files to rigion labels -------- #

def find_matching_zarrs(row):
    mask = (
        (ping_time_df['ping_start'] <= row['start_time']) &
        (ping_time_df['ping_end'] >= row['end_time'])
    )
    
    return ping_time_df.loc[mask, 'id'].tolist()

# Apply it to get a list of Zarr paths per region
region_time_df['matching_zarrs'] = region_time_df.apply(find_matching_zarrs, axis=1)

region_time_df

,id,region_id,start_time,end_time,ping_time,depth,matching_zarrs
0,x0003_0_wt_20170626_125619_f0013,0,2017-06-26 13:01:22.781000,2017-06-26 13:27:43.071000,['2017-06-26T13:01:22.781000000' '2017-06-26T1...,[174.22032918 177.11883162 196.86310824 194.69...,[x0003_0_wt_20170626_125619_f0013]
1,x0003_0_wt_20170626_125619_f0013,1,2017-06-26 13:58:29.580000,2017-06-26 14:24:16.405000,['2017-06-26T13:58:32.458000000' '2017-06-26T1...,[250.25367156 272.30974633 278.61148198 271.52...,[x0003_0_wt_20170626_125619_f0013]
2,x0004_0_wt_20170626_191344_f0006,2,2017-06-26 20:21:13.749000,2017-06-26 20:42:11.118500,['2017-06-26T20:21:48.969000000' '2017-06-26T2...,[288.75080198 287.36398207 278.5807893 275.80...,[x0004_0_wt_20170626_191344_f0006]
3,x0004_0_wt_20170626_191344_f0006,3,2017-06-26 20:42:11.118500,2017-06-26 21:23:28.006500,['2017-06-26T20:42:43.402000000' '2017-06-26T2...,[230.96663905 242.98574494 257.77849065 260.08...,[]
4,x0004_2_wt_20170627_004507_f0007,5,2017-06-27 00:28:51.374500,2017-06-27 00:45:13.311000,['2017-06-27T00:28:57.333500000' '2017-06-27T0...,[219.75994975 331.7614044 320.75548111 331.11...,[]
...,...,...,...,...,...,...,...
266,x0131_0_wt_20170904_141338_f0004,276,2017-09-04 15:02:31.003500,2017-09-04 15:02:45.599000,['2017-09-04T15:02:31.003500000' '2017-09-04T1...,[118.47798742 164.70440252 164.70440252 118.47...,[]
267,x0131_0_wt_20170904_141338_f0004,277,2017-09-04 15:04:50.926000,2017-09-04 15:05:05.567000,['2017-09-04T15:04:50.926000000' '2017-09-04T1...,[121.55974843 165.93710692 165.93710692 121.55...,[]
268,x0131_0_wt_20170904_141338_f0004,278,2017-09-04 15:05:36.848500,2017-09-04 15:05:49.235000,['2017-09-04T15:05:36.848500000' '2017-09-04T1...,[112.31446541 143.74842767 143.74842767 112.31...,[]
269,x0131_2_wt_20170904_172930_f0010,279,2017-09-04 17:51:59.891000,2017-09-04 18:40:14.303000,['2017-09-04T18:40:12.378000000' '2017-09-04T1...,[343.243687 294.74165453 295.84871354 287.76...,[]


In [ ]:
# load a sample zarr file

bucket = "agr230002-bucket01"
zarr_key = "hake_data/data_zarr/MVBS/2017/x0004_0_wt_20170626_191344_f0006.zarr"
full_path = f"s3://{bucket}/{zarr_key}"

ds = xr.open_zarr(fs.get_mapper(full_path))
ds

<xarray.Dataset> Size: 28MB
Dimensions:            (channel: 3, ping_time: 1556, depth: 759)
Coordinates:
  * channel            (channel) <U37 444B 'GPT  18 kHz 009072058c8d 1-1 ES18...
  * depth              (depth) float64 6kB 0.0 1.0 2.0 3.0 ... 756.0 757.0 758.0
  * ping_time          (ping_time) datetime64[ns] 12kB 2017-06-26T19:13:45 .....
Data variables:
    Sv                 (channel, ping_time, depth) float64 28MB dask.array<chunksize=(3, 1000, 759), meta=np.ndarray>
    frequency_nominal  (channel) float64 24B dask.array<chunksize=(3,), meta=np.ndarray>
    latitude           (ping_time) float64 12kB dask.array<chunksize=(1000,), meta=np.ndarray>
    longitude          (ping_time) float64 12kB dask.array<chunksize=(1000,), meta=np.ndarray>
Attributes:
    processing_function:          commongrid.compute_MVBS
    processing_level:             Level 3A
    processing_level_url:         https://echopype.readthedocs.io/en/stable/p...
    processing_software_name:     echopype
    processing_software_version:  0.9.0
    processing_time:              2024-08-14T01:46:38Z

In [ ]:
# ------- Function for creating echograms ------- #

def create_echogram(ds, channel):
    try:
        eg = ds.eshader.echogram(
            channel = [channel],
            cmap = "jet"
        )()
    
    except:
        eg = ds.eshader.echogram(
            vert_dim="depth",
            channel=[channel],
            cmap="jet",
        )()
    
    return eg

In [8]:
# Create echogram
channel = ds.coords['channel'].values[1]  # Get the 38kHz channel

eg = create_echogram(ds, channel)

# Create histogram
hist = ds.eshader.hist(
    bins = 20,
    overlay = True,
)

table = ds.eshader.table()

pn.Column(eg, hist, table)

BokehModel(combine_events=True, render_bundle={'docs_json': {'d7e3afc3-71cc-487c-999a-ac6caf2c09c7': {'version…

In [ ]:
# ------------------ Region Selection ------------------ #

def get_matching_region_labels(zarr_key, region_time_df):
    """Get region labels matching the zarr ID"""
    zarr_id = os.path.basename(zarr_key).replace(".zarr", "")
    
    return region_time_df[region_time_df['matching_zarrs'].apply(lambda x: zarr_id in x if isinstance(x, list) else False)]

matching_rows = get_matching_region_labels(zarr_key, region_time_df)
matching_rows

,id,region_id,start_time,end_time,ping_time,depth,matching_zarrs
2,x0004_0_wt_20170626_191344_f0006,2,2017-06-26 20:21:13.749,2017-06-26 20:42:11.118500,['2017-06-26T20:21:48.969000000' '2017-06-26T2...,[288.75080198 287.36398207 278.5807893 275.80...,[x0004_0_wt_20170626_191344_f0006]


In [ ]:
# ----------- Parse Polygons from DataFrame ----------- #

def parse_polygons_from_df(df):
    all_polygons = []

    for i, row in df.iterrows():
        try:
            # Get the raw string values
            ping_time_str = str(row['ping_time'])
            depth_str = str(row['depth'])
            
            # Extract ping times using regex
            ping_times = re.findall(r"'([^']+)'|\"([^\"]+)\"", ping_time_str)
            ping_times = [t[0] or t[1] for t in ping_times if t[0] or t[1]]
            
            # Extract depth values using regex
            depth_values = re.findall(r"(\d+\.\d+)", depth_str)
            depth_values = [float(d) for d in depth_values]
            
            # Validate lengths
            if len(ping_times) != len(depth_values) or len(ping_times) < 3:
                print(f"Skipping row {i}: mismatched length (times: {len(ping_times)}, depths: {len(depth_values)})")
                continue
            
            # Convert to datetime objects
            xs_datetime = pd.to_datetime(ping_times, errors='coerce').dropna()
            
            # Convert datetime to a format Bokeh can render (milliseconds since epoch)
            xs_epoch = xs_datetime.astype(np.int64) // 10**6
            
            # Ensure we have enough valid points after datetime conversion
            if len(xs_epoch) < 3:
                print(f"Skipping row {i}: not enough valid timestamps after conversion")
                continue
            
            # Make sure depths match remaining timestamps
            ys = depth_values[:len(xs_epoch)]
            
            # Create points as a list of (epoch time, depth) tuples
            points = list(zip(xs_epoch.tolist(), ys))
            
            # Close polygon if needed
            if points[0] != points[-1]:
                points.append(points[0])
            
            # Add to our list of polygons
            all_polygons.append(points)
            print(f"Added polygon with {len(points)} points using epoch time (ms)")
            
        except Exception as e:
            print(f"Failed to parse row {i}: {e}")
            continue

    return all_polygons

# Parse polygons
polygon_data = parse_polygons_from_df(matching_rows)

Added polygon with 51 points using epoch time (ms)


In [11]:
# If there are existing polygons, convert ping_time in ds to overlay them and create new echogram
# Create initial polygons element with existing data

poly = hv.Polygons([]).opts(
        opts.Polygons(
            fill_alpha=0.3,
            line_width=2,
            color='red',
            tools=['hover']
        )
    )

if polygon_data:
    ds = ds.assign_coords({
        'ping_time': (('ping_time',), ds['ping_time'].data.astype('int64') // 10**6),
    })

    eg =  create_echogram(ds, channel)

    poly = hv.Polygons(polygon_data).opts(
        opts.Polygons(
            fill_alpha=0.3,
            line_width=2,
            color='red',
            tools=['hover']
        )
    )

In [ ]:
# ------- Create the drawing and editing streams ------- #

poly_draw = PolyDraw(
    source=poly,
    show_vertices=True,
    drag=True,
    num_objects=50,
    vertex_style=dict(size=4, color='red')
)

poly_edit = PolyEdit(
    source=poly,
    show_vertices=True,
    vertex_style=dict(size=4, color='red'),
    shared=True
)

# Overlay the echogram with the polygon element
overlay = eg * poly

dashboard = overlay.opts(
    opts.Polygons(fill_alpha=0.3, active_tools=['poly_draw', 'poly_edit'])
)

In [13]:
# ------------------ Widgets ------------------ #
text_output = pn.widgets.StaticText(value="")
histogram_panel = pn.Column(sizing_mode='stretch_width')

In [ ]:
# Format filename based on ping_time range in ds
start_dt = pd.to_datetime(ds['ping_time'][0].values, unit='ms')
end_dt = pd.to_datetime(ds['ping_time'][-1].values, unit='ms')
start_time = start_dt.strftime("%Y%m%d_%H%M%S")
end_time = end_dt.strftime("%Y%m%d_%H%M%S")
file_name = f"{start_time}_{end_time}_regions.csv"
save_path = os.path.join(SAVE_DIR, file_name)


# ------------ Create a callback that will extract polygon data ----------- #
def export_callback(event, save_path = file_name):
    print(">> Export button clicked!")
    polygon_vertices = {
        'xs': poly_draw.data.get('xs', []),
        'ys': poly_draw.data.get('ys', [])
        }
    
    # Check if there are any polygons
    if not polygon_vertices['xs'] or not polygon_vertices['ys']:
        text_output.value = "No polygons to export. Please draw at least one polygon."
        return
    
    valid_regions = [
        (np.asarray(x), np.asarray(y))
        for x, y in zip(polygon_vertices['xs'], polygon_vertices['ys'])
        if len(x) > 2 and len(y) > 2
    ]
    
    if not valid_regions:
        text_output.value = "No valid data extracted from polygons."
        return

    # Construct rows with clean Python lists
    records = []
    for i, (x_vals, y_vals) in enumerate(valid_regions):
        records.append({
            "region_id": i,
            "time": x_vals,
            "depth": y_vals
        })


    vertices_df = pd.DataFrame(records)
    
    # Flatten the lists for the dataframe
 

    vertices_df.to_csv(save_path, index=False)
    text_output.value = f"Saved region data as: {file_name}"


In [15]:
# Export vertices button
export_button = pn.widgets.Button(name='Export Vertices', button_type='primary')
export_button.on_click(export_callback)

Watcher(inst=Button(button_type='primary', name='Export Vertices'), cls=<class 'panel.widgets.button.Button'>, fn=<function export_callback at 0x321bb4cc0>, mode='args', onlychanged=False, parameter_names=('clicks',), what='value', queued=False, precedence=0)

In [16]:
# ---------- Function to extract data from polygons ---------- #

def extract_data_from_polygon(polygon_vertices, ds = ds, save=False):
    # Convert dataset into DataFrame and extract needed columns
    df = ds.to_dataframe().reset_index()

    # Handle different depth column names
    if 'echo_range' in df.columns:
        depth_col = 'echo_range'
    elif 'depth' in df.columns:
        depth_col = 'depth'
    else:
        text_output.value = "Missing both 'echo_range' and 'depth' columns"
        return {}

    df = df[['channel', 'ping_time', depth_col, 'Sv']].copy()
    df.rename(columns={depth_col: 'echo_range'}, inplace=True)


    # Ensure ping_time is datetime
    if not np.issubdtype(df['ping_time'].dtype, np.datetime64):
        df['ping_time'] = pd.to_datetime(df['ping_time'], unit='ms')
    
    # Convert 'ping_time' to ns for plotting
    df['ping_time_numeric'] = pd.to_datetime(df['ping_time']).astype('int64') // 10**6

    # Create GeoDataFrame with points
    df['geometry'] = gpd.points_from_xy(df['ping_time_numeric'], df['echo_range'])
    gdf = gpd.GeoDataFrame(df, geometry='geometry')

    # Dictionary to store extracted data
    extracted_data = {}

    # Iterate over each polygon in the drawn data
    for i, (xs, ys) in enumerate(zip(polygon_vertices['xs'], polygon_vertices['ys'])):
        if len(xs) <= 2 or len(ys) <= 2:
            text_output.value = f"Skipping region {i+1}: needs at least 3 vertices"
            continue
            
        text_output.value = f"Processing region {i+1}..."
        
        # Create shapely Polygon
        polygon = Polygon(zip(xs, ys))
        
        # Get points inside the polygon using vectorized operations
        inside_mask = gdf['geometry'].within(polygon)
        
        if inside_mask.sum() == 0:
            text_output.value = f"No data points found in region {i+1}"
            continue
            
        filtered_df = gdf.loc[inside_mask, ['channel', 'ping_time', 'echo_range', 'Sv']]
        
        # Save to dictionary
        region_name = f"region_{i+1}"
        save_path = os.path.join(SAVE_DIR, f"{region_name}.csv")
        extracted_data[region_name] = filtered_df

        text_output.value = f"{region_name}: {inside_mask.sum()} points inside the selected polygon"

        # Optionally save each region to a CSV file
        if save:
            filtered_df.to_csv(save_path, index=False)
            text_output.value = f"Saved {region_name} at: {save_path}"

    return extracted_data


In [17]:
# ---------- Function to create and display histograms for polygon regions ---------- #

def display_histograms_callback(event):
    # Clear previous histograms
    histogram_panel.clear()
    text_output.value = "Generating histogram..."

    # Get polygon vertices
    polygon_vertices = {
        'xs': poly_draw.data.get('xs', []),
        'ys': poly_draw.data.get('ys', [])
    }
    
    # Check if there are any polygons
    if not polygon_vertices['xs'] or not polygon_vertices['ys']:
        text_output.value = "No polygons to analyze. Please draw at least one polygon."
        return
    
    # Extract data from polygons
    extracted_data = extract_data_from_polygon(polygon_vertices)
    
    # Check if the there is any valid data
    if not extracted_data:
        text_output.value = "No valid data extracted from polygons."
        return
    
    text_output.value = f"Creating histogram for {len(extracted_data)} regions..."
    
    # Create histograms for each region
    for region_name, region_data in extracted_data.items():
        if len(region_data) == 0:
            continue
            
        # Create histogram using HoloViews
        sv_values = region_data['Sv'].dropna().values
        if len(sv_values) == 0:
            continue
            
        hist_obj = hv.Histogram(np.histogram(sv_values, bins=20))
        hist_plot = hist_obj.opts(
            title=f"{channel} Sv Values - {region_name}",
            xlabel='Sv (dB)',
            ylabel='Frequency',
            height=250,
            width=500,
            color='blue'
        )
        
        # Add summary statistics 
        stats_text = f"""
            <b>Region:</b> {region_name}<br>
            <b>Count:</b> {len(sv_values)}<br>
            <b>Mean:</b> {np.mean(sv_values):.2f} dB<br>
            <b>Median:</b> {np.median(sv_values):.2f} dB<br>
            <b>Min:</b> {np.min(sv_values):.2f} dB<br>
            <b>Max:</b> {np.max(sv_values):.2f} dB<br>
            <b>Std Dev:</b> {np.std(sv_values):.2f} dB
        """
        
        stats_widget = pn.widgets.StaticText(value=stats_text)
        
        # Add to the histogram panel
        histogram_panel.append(pn.Row(hist_plot, stats_widget))
    
    text_output.value = f"Generated histograms for {len(extracted_data)} regions"

In [18]:
# Create button for displaying histograms
histogram_button = pn.widgets.Button(name='Display Histograms', button_type='primary')
histogram_button.on_click(display_histograms_callback)

Watcher(inst=Button(button_type='primary', name='Display Histograms'), cls=<class 'panel.widgets.button.Button'>, fn=<function display_histograms_callback at 0x30ff902c0>, mode='args', onlychanged=False, parameter_names=('clicks',), what='value', queued=False, precedence=0)

### PolyDraw

**Add patch/multi-line**

- Double tap to add the first vertex, then use tap to add each subsequent vertex, to finalize the draw action double tap to insert the final vertex or press the ESC key to stop drawing.

**Move patch/multi-line**

- Tap and drag an existing patch/multi-line; the point will be dropped once you let go of the mouse button.

**Delete patch/multi-line**

- Tap a patch/multi-line to select it then press BACKSPACE key while the mouse is within the plot area



### PolyEdit
**Show vertices**

- Long tap an existing patch or multi-line

**Add vertex**

- Double tap an existing vertex to select it, the tool will draw the next point, to add it tap in a new location.
- To finish editing and add a point double tap otherwise press the ESC key to cancel.

**Move vertex**

- Drag an existing vertex and let go of the mouse button to release it.

**Delete vertex**

- After selecting one or more vertices press BACKSPACE while the mouse cursor is within the plot area

In [21]:
# Initialize text and clear histograms on startup
text_output.value = "Draw polygons on the echogram"
histogram_panel.clear()

# Update layout to include the new histogram button and panel
layout = pn.Column(
    pn.Row(
        pn.Column(
            dashboard,
            pn.Row(export_button, histogram_button),
            text_output
        ),
    ),
    histogram_panel
)

# Display the layout
layout.servable()

BokehModel(combine_events=True, render_bundle={'docs_json': {'508ddc9a-8841-44be-a89a-78c3fb3bf6e9': {'version…